**makemore: becoming a backprop ninja**

完全使用之前 `4_MLP2_makemore.ipynb` 中的单个隐藏层的多层感知机（关于为什么是"单层", 详见： `4_MLP2_makemore.ipynb`中的 **✅总结** 部分 ）

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27
torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# 功能函数 用来比较梯度
# 主要是比较自己手动实现的和pytorch接口的梯度结果是否相同

def cmp(s,dt,t):
    """
    s: 要比较的梯度对象的名称 字符串
    dt: dloss/dt的梯度结果
    t: torch张量对象
    """
    ex = torch.all(dt == t.grad).item()  # exact 精确比较
    app = torch.allclose(dt == t.grad)   # approximate 近似比较
    maxdiff = (dt - t.grad).abs().max().item()   # 张量中最大梯度值差异
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {str(maxdiff):5s}")

In [4]:
# 初始化
n_embd = 10
n_hidden = 64 

g = torch.Generator().manual_seed(2147483647) 
# 输入层
C  = torch.randn((vocab_size, n_embd), generator=g)

# Layer1 隐藏层
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)*(5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g)*0.1

# Layer2 输出层
W2 = torch.randn((n_hidden, vocab_size), generator=g)*0.1
b2 = torch.randn(vocab_size,  generator=g)*0.1

# BN层
bngain = torch.randn(n_hidden)*0.1 + 1.0
bnbias = torch.randn(n_hidden)*0.1

# 注意，通常会将bias都初始化为0，但这里都初始化为比较小的数字，而不是精确的0
# 这是因为如果初始化为0，可能会掩盖梯度计算实现过程中的错误，所以这里改成比较小的小数，方便纠错自查
# 另外这里加了BN但是还在线性层用了bias，是因为只是想检查梯度的计算结果，并不是和网络训练相关

parameters = [C, W1,b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

4137


In [5]:
batch_size = 32
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

$$ Z = \frac{X - \mu}{\sigma} = \frac{X - \frac{1}{n} \sum_{i=1}^{n} x_i}{\sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2}} = \frac{X - \bar{x}}{\sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2}}$$


In [ ]:
# forward pass
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1) 
# Linear Layer
hprebn = torch.tanh(embcat @ W1 + b1)    # hidden layer pre-activation
# BatchNorm Layer
bnmeani = (1/n)*hprebn.sum(dim = 0, keepdim=True) # bessel correction 贝塞尔纠正 (1/(n-1))
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = (1/n)*bndiff2.sum(dim =0, keepdim = True) # 这里关于均值和方差的计算公式 详见: 4_MLP2_makemore.ipynb 中的  对正态分布进行标准化的公式

# Linear Layer 2
logits = h @ W2 + b2 
# loss function
loss = F.cross_entropy(logits, Yb)

# backward pass
for p in parameters:
    p.grad = None